# Models Training with their Best Parameters

## Setup

### Imports

In [ ]:
import pickle
import pathlib
import warnings
import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.base import BaseEstimator
from sklearn.model_selection import KFold, cross_val_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

### Datasets

In [ ]:
PREPROCESSED_CREDIT_RISK_DATASET = pathlib.Path('../datasets/credit_risk.pkl')

with open(PREPROCESSED_CREDIT_RISK_DATASET, mode="rb") as f:
    X_credit_risk_train, X_credit_risk_test, y_credit_risk_train, y_credit_risk_test = pickle.load(f)

X_credit_risk = np.concatenate((X_credit_risk_train, X_credit_risk_test), axis=0)
y_credit_risk = np.concatenate((y_credit_risk_train, y_credit_risk_test), axis=0)

print(f"X shape: {X_credit_risk.shape}")
print(f"y shape: {y_credit_risk.shape}")

## Training

### Models Array

In [ ]:
models:list[BaseEstimator] = [
    DecisionTreeClassifier(criterion='entropy', min_samples_leaf=1, min_samples_split=5, splitter='best'),
    RandomForestClassifier(criterion='entropy', min_samples_leaf=1, min_samples_split=5, n_estimators=40),
    KNeighborsClassifier(n_neighbors=20, p=1),
    LogisticRegression(C=1.0, solver='lbfgs', tol=0.0001),
    SVC(C=1.5, kernel='rbf', tol=0.001),
    MLPClassifier(activation='tanh', batch_size=10, hidden_layer_sizes=(2,2), solver='adam')
]

### Multiple Cross Validations

#### Auxiliar function

In [ ]:
def apply_cross_validations_in_models(
    models:list[BaseEstimator], 
    X:np.typing.ArrayLike, 
    y:np.typing.ArrayLike,
    n_splits:int = 10,
    n_tests:int = 30
) -> pd.DataFrame:
    results:dict[str, list[float]] = {model.__class__.__name__: [] for model in models}

    for i in tqdm(range(n_tests), desc="Tests", position=0):
        k_fold = KFold(n_splits=n_splits, shuffle=True, random_state=i)

        for model in models:
            scores = cross_val_score(estimator=model, X=X, y=y, cv=k_fold)
            results[model.__class__.__name__].append(scores.mean())

    return pd.DataFrame(results)

#### Evaluating and Saving the Results

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    df = apply_cross_validations_in_models(
        models=models,
        X=X_credit_risk,
        y=y_credit_risk
    )

    display(df)
    df.to_csv('results.csv', index=False)
